# 02 · Modelado — ALS (Alternating Least Squares)

Este notebook entrena un modelo de **factorización de matrices con feedback implícito** usando
[`implicit`](https://github.com/benfred/implicit)

El algoritmo ALS (Alternating Least Squares), o Mínimos Cuadrados Alternos, es uno de los métodos más potentes en la construcción de motores de recomendación basados en filtrado colaborativo. Es el motor matemático detrás de las sugerencias personalizadas de gigantes tecnológicos como Spotify y Netflix.


In [1]:
import polars as pl
import numpy as np
from scipy.sparse import csr_matrix
from pathlib import Path
import time

DATA_DIR = Path(r"datasets/parquet/steam/reviews/processed")

df = pl.read_parquet(DATA_DIR / "steam_reviews_clean.parquet")
user_map = pl.read_parquet(DATA_DIR / "user_map.parquet")
item_map = pl.read_parquet(DATA_DIR / "item_map.parquet")

n_users = user_map.height
n_items = item_map.height
print(f"Filas: {df.height:,} | usuarios: {n_users:,} | juegos: {n_items:,}")
df.head()

Filas: 92,958,418 | usuarios: 17,049,073 | juegos: 78,376


user_idx,item_idx,recommended,timestamp
i32,i32,i8,i64
1713753,12577,1,1575375715
694711,12614,1,1543113900
11632343,24377,1,1606781931
3431601,5925,1,1611595804
1446995,25836,1,1561910227


## 1. Train / test split

Se realiza una separacion de datos en train y test, pero no de forma global sino por usuario.<br>
Esto significa que tomamos cada usuario y dejamos un review por fuera de los datos de entrenamiento.<br>
En la etapa de preparación de datos nos aseguramos de incluir solamente usuarios con 2 o mas reviews, por lo que podemos realizar esta separación sin problema.<br>
Este split nos permite evaluar si el modelo es capaz de predecir este review que quedó por fuera, y que tan bien posicionada (top) se encuentra esta recomendación.

In [2]:
# Si se cuenta con la variable timestamp, se utiliza para dejar como Test el review mas reciente, caso contrario toma uno al azar.
has_timestamp = "timestamp" in df.columns

if has_timestamp:
    # última interacción de cada usuario (por timestamp) -> test
    df = df.with_columns(
        pl.col("timestamp").rank(method="ordinal", descending=True).over("user_idx").alias("rank_in_user")
    )
else:
    # sin timestamp confiable: tomamos una interacción al azar por usuario para test (fijamos seed)
    df = df.with_columns(pl.int_range(pl.len()).shuffle(seed=42).over("user_idx").alias("rank_in_user") + 1)

test_df = df.filter(pl.col("rank_in_user") == 1).drop("rank_in_user")
train_df = df.filter(pl.col("rank_in_user") != 1).drop("rank_in_user")

print(f"Train: {train_df.height:,} filas | Test: {test_df.height:,} filas")

Train: 75,909,345 filas | Test: 17,049,073 filas


## 2. Matrices sparse (CSR)

La librería `implicit` espera una matriz `usuarios x items` en formato CSR, con los valores establecidos como confianza.
Para feedback implícito, lo estándar (paper original de Hu, Koren & Volinsky) es transformar la señal binaria en:

```
confianza = 1 + alpha * señal
```

donde `alpha` controla cuánto más confiamos en una interacción positiva registrada vs. el "no sabemos" implícito de
lo que el usuario no tiene review.<br>
Este es un hiperparametro que se revisará posteriormente, por ahora se utiliza un valor `alpha=40`

In [3]:
ALPHA = 40.0

In [4]:
# En esta iteración el enfoque es "Implicit", tomamos unicamente el feedback positivo, es decir recommended==1.
# Bajo el modelo implicito, un feedback negativo tendrá valor de confianza = 1 + alpha * 0, es decir la menor confianza
# Esto significa que un negativo se interpreta de la misma forma que un desconocido.
# En etapas posteriores se evaluará un modelo "Explicit", el cual esta diseñado para considerar bajos reviews (0-5 estrellas, like/dislike)

def build_csr(data: pl.DataFrame, n_users: int, n_items: int, alpha: float = ALPHA) -> csr_matrix:
    # Filtra solo reviews positivas
    pos = data.filter(pl.col("recommended") == 1)
    # Tablas de usuarios / items
    rows = pos["user_idx"].to_numpy()
    cols = pos["item_idx"].to_numpy()
    # Formula de confianza
    vals = np.full(len(pos), 1.0 + alpha, dtype=np.float32)
    return csr_matrix((vals, (rows, cols)), shape=(n_users, n_items))

In [4]:
t0 = time.time()
train_matrix = build_csr(train_df, n_users, n_items)
print(f"Matriz train: {train_matrix.shape}, nnz={train_matrix.nnz:,}, construida en {time.time()-t0:.1f}s")
print(f"Memoria aprox: {(train_matrix.data.nbytes + train_matrix.indices.nbytes + train_matrix.indptr.nbytes)/1e6:.1f} MB")

Matriz train: (17049073, 78376), nnz=65,077,598, construida en 4.6s
Memoria aprox: 588.8 MB


## 3. Entrenamiento ALS (CPU, multi-hilo)

Hiperparámetros de partida razonables para este tamaño de dataset. `num_threads=0` usa todos los cores disponibles.

In [6]:
# Libreria implicit, Licencia MIT Copyright (c) 2016 Ben Frederickson
# https://github.com/benfred/implicit.git
from implicit.als import AlternatingLeastSquares
from threadpoolctl import threadpool_limits

# Limita BLAS a 1 hilo para evitar conflictos
threadpool_limits(1, "blas")

model = AlternatingLeastSquares(
    factors=128,
    regularization=0.05,
    iterations=20,
    num_threads=0,     # 0 = todos los cores
    random_state=42,
)

t0 = time.time()
model.fit(train_matrix, show_progress=True)
print(f"\nEntrenamiento completo en {(time.time()-t0)/60:.1f} min")

  0%|          | 0/20 [00:00<?, ?it/s]


Entrenamiento completo en 10.2 min


## 4. Evaluación: Hit Rate@K y MRR@K

**Hit Rate** (Tasa de Aciertos):<br>
Mide si el sistema fue capaz de adivinar al menos un producto que le interesa al usuario dentro de la lista de recomendaciones.<br>
**MRR** (Rango Recíproco Medio):<br>
Mide qué tan arriba en la lista aparece la primera recomendación correcta. Evalúa la posición (el orden)



Contra la última interacción positiva real de cada usuario que se dejo en Test, medimos si el modelo la ubica entre
sus top-K recomendaciones.

In [5]:
# Funcion para realizar la evaluación del modelo. Define una muestra de usuarios
def evaluate(model, train_matrix, test_df, k=10, sample_users=20000, seed=42):
    test_pos = test_df.filter(pl.col("recommended") == 1)
    users_with_test = test_pos["user_idx"].unique().to_numpy()

    rng = np.random.default_rng(seed)
    if len(users_with_test) > sample_users:
        users_with_test = rng.choice(users_with_test, size=sample_users, replace=False)

    test_lookup = dict(zip(test_pos["user_idx"].to_list(), test_pos["item_idx"].to_list()))

    hits = 0
    reciprocal_ranks = []

    ids, _ = model.recommend(
        users_with_test,
        train_matrix[users_with_test],
        N=k,
        filter_already_liked_items=True,
    )

    for row, uidx in enumerate(users_with_test):
        true_item = test_lookup[uidx]
        recs = ids[row]
        if true_item in recs:
            hits += 1
            rank = int(np.where(recs == true_item)[0][0]) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    precision_at_k = hits / len(users_with_test)
    mrr_at_k = float(np.mean(reciprocal_ranks))
    return precision_at_k, mrr_at_k, len(users_with_test)

In [9]:
t0 = time.time()
p_at_10, mrr_at_10, n_eval = evaluate(model, train_matrix, test_df, k=10)
print(f"Evaluado sobre {n_eval:,} usuarios en {time.time()-t0:.1f}s")
print(f"Hit Rate@10 (precision@10 sobre 1 item real): {p_at_10:.4f}")
print(f"MRR@10: {mrr_at_10:.4f}")

Evaluado sobre 20,000 usuarios en 19.8s
Hit Rate@10 (precision@10 sobre 1 item real): 0.1060
MRR@10: 0.0475


**Modelo 0:**<br>
Evaluando Hit Rate, el 10.6% de los usuarios obtuvieron recomendaciones de 1 o mas juegos que ellos mismos valoraron de forma positiva posteriormente<br>
Para fines de comparación de este modelo inicial versus el simple azar, si de entre el total de juegos (78,376) tomamos 10 de forma aleatoria, la probabilidad de acertar nuestra recomendación seria de:<br>
> 10 / 78,376 = 0.0001276  →  0.0128%

Si comparamos el valor → 0.0128% de la distribución aleatoria con los resultados del Modelo 0 → 10.60%, sin optimizaciónes de hiperparametros se obtiene:
> 0.1060 / 0.0001276 ≈ 830 veces mejor que adivinar al azar.



Un detalle importante sobre el MRR, normalmente el calculo es sobre los aciertos, que tan arriba del top 10 se posicionaron.<br>
En este caso el calculo de MRR es global, es decir: se promedia sobre los N usuarios evaluados en total (no solo sobre los que tuvieron acierto) — los que no aciertan dentro del top-K aportan un 0 al promedio, no se excluyen del denominador.<br>

Si quisieramos obtener el MRR condicional (solo de los aciertos) la formula seria el MRR Global / Hit Rate:
> 0.0475 / 0.1060 → 0.4481

El inverso del MRR Condicional (1/0.4481) se obtiene 2.23 que representa la posicion promedio dentro del top 10 de juegos recomendados, cuando hay acierto.<br>
Esta metrica es bastante buena porque significa que cuando acertamos, la recomendacion esta entre los primeros lugares, y no al final de la lista.

**Versus recomendar solo el top 10 siempre?**<br>
Una evaluación importante es comparar el modelo personalizado contra un modelo "tonto" que solo recomienda el top 10 global siempre, indistinto de la persona.<br>
Partiendo de la hoja anterior (ETL) se observa un gran sesgo con los juegos mas populares, si bien es normal que el modelo recomiende estos juegos (por algo son populares), como mínimo esperariamos que las recomendaciones sean mejores que un simple top 10.<br>
El siguiente bloque de codigo obtiene las metricas que generaría este modelo que solo conoce 10 juegos.

In [21]:
top_items_pop = (
    train_df.filter(pl.col("recommended") == 1)
      .group_by("item_idx")
      .agg(pl.len().alias("n"))
      .sort("n", descending=True)
      .head(10)["item_idx"]
      .to_numpy()
)

test_pos = test_df.filter(pl.col("recommended") == 1)
test_lookup = dict(zip(test_pos["user_idx"].to_list(), test_pos["item_idx"].to_list()))

hits_pop, rr_pop = 0, []
for uidx, true_item in test_lookup.items():
    if true_item in top_items_pop:
        hits_pop += 1
        rank = int(np.where(top_items_pop == true_item)[0][0]) + 1
        rr_pop.append(1.0 / rank)
    else:
        rr_pop.append(0.0)

print(f"Popularidad (catálogo completo) — Hit Rate@10: {hits_pop/len(test_lookup):.4f}")
print(f"Popularidad (catálogo completo) — MRR@10: {np.mean(rr_pop):.4f}")

Popularidad (catálogo completo) — Hit Rate@10: 0.1190
Popularidad (catálogo completo) — MRR@10: 0.0563


Tal como se observa, una recomendación plana de los 10 juegos mas populares es ligeramente superior en terminos de precisión versus el modelo base<br>
> HitRate: 0.1190 vs 0.1060<br>
> MRR@10 : 0.0563 vs 0.0475

Esto no significa que el modelo este mal, simplemente evidencia el enorme desvalance que hay en los juegos top versus los juegos niche.<br>
Igualmente, es un buen punto de partida para determinar si el modelo final supera una recomendación genérica.

## 5. Recomendaciones de ejemplo

In [12]:
item_lookup = dict(zip(item_map["item_idx"].to_list(), item_map["item_id"].to_list()))
user_lookup = dict(zip(user_map["user_idx"].to_list(), user_map["user_id"].to_list()))

sample_user_idx = int(train_df["user_idx"][0])
sample_user_id = user_lookup[sample_user_idx]

ids, scores = model.recommend(sample_user_idx, train_matrix[sample_user_idx], N=10)

print(f"Recomendaciones para usuario {sample_user_id} (idx={sample_user_idx}):\n")
for item_idx, score in zip(ids, scores):
    print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}")

Recomendaciones para usuario 76561197995052837 (idx=694711):

  app_id=1086940      score=0.8953
  app_id=1172620      score=0.6483
  app_id=1282100      score=0.6220
  app_id=1151340      score=0.6037
  app_id=1716740      score=0.5999
  app_id=680420       score=0.5751
  app_id=597820       score=0.5676
  app_id=1868140      score=0.5672
  app_id=292030       score=0.5439
  app_id=1085660      score=0.5299


In [17]:
item_lookup = dict(zip(item_map["item_idx"].to_list(), item_map["item_id"].to_list()))
user_lookup = dict(zip(user_map["user_idx"].to_list(), user_map["user_id"].to_list()))
user_idx_lookup = {v: k for k, v in user_lookup.items()}
item_idx_lookup = {v: k for k, v in item_lookup.items()}

TARGET_USER_ID = 76561198324251650

if TARGET_USER_ID not in user_idx_lookup:
    print(f"⚠️  El usuario {TARGET_USER_ID} no está en el subset de juegos
    print("   o por tener menos de MIN_REVIEWS_POR_USUARIO_JUEGOS reviews de juegos).")
else:
    sample_user_idx = user_idx_lookup[TARGET_USER_ID]
    ids, scores = model.recommend(sample_user_idx, train_matrix[sample_user_idx], N=10)
    print(f"Recomendaciones para usuario {TARGET_USER_ID}:\n")
    for item_idx, score in zip(ids, scores):
        print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}")

Recomendaciones para usuario 76561198324251650:

  app_id=1113560      score=0.3609
  app_id=578080       score=0.3081
  app_id=814380       score=0.2568
  app_id=678960       score=0.2388
  app_id=637650       score=0.2351
  app_id=440          score=0.2292
  app_id=431960       score=0.2125
  app_id=39210        score=0.1970
  app_id=447530       score=0.1833
  app_id=460790       score=0.1777


In [15]:
# Juegos similares a uno dado (útil para "usuarios que jugaron esto también jugaron...")
sample_item_idx = int(train_df["item_idx"][0])
sample_item_id = item_lookup[sample_item_idx]

similar_ids, similar_scores = model.similar_items(sample_item_idx, N=10)

print(f"Juegos similares a app_id={sample_item_id} (idx={sample_item_idx}):\n")
for item_idx, score in zip(similar_ids, similar_scores):
    print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}")

Juegos similares a app_id=435150 (idx=12614):

  app_id=435150       score=1.0000
  app_id=373420       score=0.5555
  app_id=560130       score=0.4360
  app_id=1086940      score=0.4341
  app_id=1184370      score=0.4203
  app_id=632470       score=0.3908
  app_id=640820       score=0.3489
  app_id=880640       score=0.3208
  app_id=362960       score=0.3119
  app_id=719040       score=0.3112


In [18]:
TARGET_APP_ID = 489830 #Skyrim

if TARGET_APP_ID not in item_idx_lookup:
    print(f"⚠️  El app_id={TARGET_APP_ID} no está en el subset de juegos.")
else:
    sample_item_idx = item_idx_lookup[TARGET_APP_ID]
    similar_ids, similar_scores = model.similar_items(sample_item_idx, N=10)
    print(f"Juegos similares (solo juegos) a app_id={TARGET_APP_ID}:\n")
    for item_idx, score in zip(similar_ids, similar_scores):
        print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}")

Juegos similares (solo juegos) a app_id=489830:

  app_id=489830       score=1.0000
  app_id=1746860      score=0.3810
  app_id=377160       score=0.3398
  app_id=22330        score=0.3236
  app_id=900883       score=0.3232
  app_id=976620       score=0.3004
  app_id=306130       score=0.2966
  app_id=204360       score=0.2763
  app_id=22320        score=0.2517
  app_id=365720       score=0.2292


## 6. Guardar el modelo

In [20]:
MODEL_DIR = Path(r"models/Raw")
MODEL_DIR.mkdir(exist_ok=True)

model.save(str(MODEL_DIR / "als_model.npz"))
print("Modelo guardado en:", MODEL_DIR / "als_model.npz")

# Para cargarlo después:
# from implicit.als import AlternatingLeastSquares
# model = AlternatingLeastSquares.load(str(MODEL_DIR / "als_model.npz"))

Modelo guardado en: models/Raw/als_model.npz


## 7. Hyperparameter Tunning
Se proceden a probar distintos hiperparametros, uno a la vez, a fin de determinar los valores que mejor se ajustan a los datos.

In [9]:
resultados_alpha = []
for alpha in [5, 10, 20, 40, 80]:
    tm = build_csr(train_df, n_users, n_items, alpha=alpha)
    m = AlternatingLeastSquares(factors=128, regularization=0.05, iterations=15, num_threads=0, random_state=42)
    m.fit(tm, show_progress=False)
    hr, mrr, _ = evaluate(m, tm, test_df, k=10)
    resultados_alpha.append({"alpha": alpha, "hit_rate": hr, "mrr": mrr})
    print(f"alpha={alpha:>4} -> Hit Rate@10: {hr:.4f} | MRR@10: {mrr:.4f}")

pl.DataFrame(resultados_alpha)

alpha=   5 -> Hit Rate@10: 0.0800 | MRR@10: 0.0386
alpha=  10 -> Hit Rate@10: 0.0894 | MRR@10: 0.0427
alpha=  20 -> Hit Rate@10: 0.0986 | MRR@10: 0.0446
alpha=  40 -> Hit Rate@10: 0.1027 | MRR@10: 0.0463
alpha=  80 -> Hit Rate@10: 0.0999 | MRR@10: 0.0450


alpha,hit_rate,mrr
i64,f64,f64
5,0.08005,0.03863
10,0.0894,0.04273
20,0.0986,0.044622
40,0.10275,0.04632
80,0.0999,0.045016


In [10]:
# Tuneo fino de Alpha
resultados_alpha_fino = []
for alpha in [30, 40, 50, 60]:
    tm = build_csr(train_df, n_users, n_items, alpha=alpha)
    m = AlternatingLeastSquares(factors=128, regularization=0.05, iterations=15, num_threads=0, random_state=42)
    m.fit(tm, show_progress=False)
    hr, mrr, _ = evaluate(m, tm, test_df, k=10)
    resultados_alpha_fino.append({"alpha": alpha, "hit_rate": hr, "mrr": mrr})
    print(f"alpha={alpha:>4} -> Hit Rate@10: {hr:.4f} | MRR@10: {mrr:.4f}")

pl.DataFrame(resultados_alpha_fino)

alpha=  30 -> Hit Rate@10: 0.1038 | MRR@10: 0.0461
alpha=  40 -> Hit Rate@10: 0.1027 | MRR@10: 0.0463
alpha=  50 -> Hit Rate@10: 0.1021 | MRR@10: 0.0462
alpha=  60 -> Hit Rate@10: 0.1021 | MRR@10: 0.0459


alpha,hit_rate,mrr
i64,f64,f64
30,0.10385,0.046099
40,0.10275,0.04632
50,0.10205,0.046222
60,0.10205,0.045936


In [7]:
# Tuneo fino de ALpha
resultados_alpha_fino = []
for alpha in [32, 35, 38]:
    tm = build_csr(train_df, n_users, n_items, alpha=alpha)
    m = AlternatingLeastSquares(factors=128, regularization=0.05, iterations=15, num_threads=0, random_state=42)
    m.fit(tm, show_progress=False)
    hr, mrr, _ = evaluate(m, tm, test_df, k=10)
    resultados_alpha_fino.append({"alpha": alpha, "hit_rate": hr, "mrr": mrr})
    print(f"alpha={alpha:>4} -> Hit Rate@10: {hr:.4f} | MRR@10: {mrr:.4f}")

pl.DataFrame(resultados_alpha_fino)

alpha=  32 -> Hit Rate@10: 0.1042 | MRR@10: 0.0464
alpha=  35 -> Hit Rate@10: 0.1023 | MRR@10: 0.0465
alpha=  38 -> Hit Rate@10: 0.1032 | MRR@10: 0.0466


alpha,hit_rate,mrr
i64,f64,f64
32,0.10415,0.046369
35,0.10235,0.046527
38,0.10325,0.046551


In [8]:
BEST_ALPHA = 32

resultados_factors = []
for factors in [64, 128, 256]:
    tm = build_csr(train_df, n_users, n_items, alpha=BEST_ALPHA)
    m = AlternatingLeastSquares(factors=factors, regularization=0.05, iterations=15, num_threads=0, random_state=42)
    m.fit(tm, show_progress=False)
    hr, mrr, _ = evaluate(m, tm, test_df, k=10)
    resultados_factors.append({"factors": factors, "hit_rate": hr, "mrr": mrr})
    print(f"factors={factors:>4} -> Hit Rate@10: {hr:.4f} | MRR@10: {mrr:.4f}")

pl.DataFrame(resultados_factors)

factors=  64 -> Hit Rate@10: 0.1172 | MRR@10: 0.0496
factors= 128 -> Hit Rate@10: 0.1042 | MRR@10: 0.0464
factors= 256 -> Hit Rate@10: 0.0839 | MRR@10: 0.0405


factors,hit_rate,mrr
i64,f64,f64
64,0.11715,0.04964
128,0.10415,0.046369
256,0.0839,0.040541


In [7]:
BEST_ALPHA = 32

resultados_factors_bajo = []
for factors in [16, 32, 48, 64]:
    tm = build_csr(train_df, n_users, n_items, alpha=BEST_ALPHA)
    m = AlternatingLeastSquares(factors=factors, regularization=0.05, iterations=15, num_threads=0, random_state=42)
    m.fit(tm, show_progress=False)
    hr, mrr, _ = evaluate(m, tm, test_df, k=10)
    resultados_factors_bajo.append({"factors": factors, "hit_rate": hr, "mrr": mrr})
    print(f"factors={factors:>4} -> Hit Rate@10: {hr:.4f} | MRR@10: {mrr:.4f}")

pl.DataFrame(resultados_factors_bajo)

factors=  16 -> Hit Rate@10: 0.1288 | MRR@10: 0.0530
factors=  32 -> Hit Rate@10: 0.1293 | MRR@10: 0.0533
factors=  48 -> Hit Rate@10: 0.1244 | MRR@10: 0.0501
factors=  64 -> Hit Rate@10: 0.1172 | MRR@10: 0.0496


factors,hit_rate,mrr
i64,f64,f64
16,0.12885,0.052957
32,0.1293,0.05333
48,0.12435,0.050051
64,0.11715,0.04964


In [8]:
BEST_ALPHA = 32

resultados_factors_fino = []
for factors in [8, 12, 16, 24]:
    tm = build_csr(train_df, n_users, n_items, alpha=BEST_ALPHA)
    m = AlternatingLeastSquares(factors=factors, regularization=0.05, iterations=15, num_threads=0, random_state=42)
    m.fit(tm, show_progress=False)
    hr, mrr, _ = evaluate(m, tm, test_df, k=10)
    resultados_factors_fino.append({"factors": factors, "hit_rate": hr, "mrr": mrr})
    print(f"factors={factors:>4} -> Hit Rate@10: {hr:.4f} | MRR@10: {mrr:.4f}")

pl.DataFrame(resultados_factors_fino)

factors=   8 -> Hit Rate@10: 0.1212 | MRR@10: 0.0502
factors=  12 -> Hit Rate@10: 0.1313 | MRR@10: 0.0500
factors=  16 -> Hit Rate@10: 0.1288 | MRR@10: 0.0530
factors=  24 -> Hit Rate@10: 0.1303 | MRR@10: 0.0529


factors,hit_rate,mrr
i64,f64,f64
8,0.12125,0.050244
12,0.13135,0.050041
16,0.12885,0.052957
24,0.1303,0.052876


In [9]:
BEST_ALPHA = 32

resultados_factors_final = []
for factors in [12, 16, 20, 24]:
    tm = build_csr(train_df, n_users, n_items, alpha=BEST_ALPHA)
    m = AlternatingLeastSquares(factors=factors, regularization=0.05, iterations=15, num_threads=0, random_state=42)
    m.fit(tm, show_progress=False)
    hr, mrr, n_eval = evaluate(m, tm, test_df, k=10, sample_users=100000)  # subimos de 20k a 100k
    resultados_factors_final.append({"factors": factors, "hit_rate": hr, "mrr": mrr})
    print(f"factors={factors:>4} -> Hit Rate@10: {hr:.4f} | MRR@10: {mrr:.4f} (n={n_eval:,})")

pl.DataFrame(resultados_factors_final)

factors=  12 -> Hit Rate@10: 0.1289 | MRR@10: 0.0496 (n=100,000)
factors=  16 -> Hit Rate@10: 0.1274 | MRR@10: 0.0513 (n=100,000)
factors=  20 -> Hit Rate@10: 0.1253 | MRR@10: 0.0487 (n=100,000)
factors=  24 -> Hit Rate@10: 0.1277 | MRR@10: 0.0519 (n=100,000)


factors,hit_rate,mrr
i64,f64,f64
12,0.12894,0.04957
16,0.12742,0.051287
20,0.12531,0.048715
24,0.12765,0.051943


In [10]:
# Regularizacion
BEST_ALPHA = 32
BEST_FACTORS = 24

resultados_reg = []
for reg in [0.01, 0.05, 0.1, 0.2]:
    tm = build_csr(train_df, n_users, n_items, alpha=BEST_ALPHA)
    m = AlternatingLeastSquares(factors=BEST_FACTORS, regularization=reg, iterations=15, num_threads=0, random_state=42)
    m.fit(tm, show_progress=False)
    hr, mrr, n_eval = evaluate(m, tm, test_df, k=10, sample_users=100000)
    resultados_reg.append({"regularization": reg, "hit_rate": hr, "mrr": mrr})
    print(f"reg={reg:>5} -> Hit Rate@10: {hr:.4f} | MRR@10: {mrr:.4f} (n={n_eval:,})")

pl.DataFrame(resultados_reg)

reg= 0.01 -> Hit Rate@10: 0.1254 | MRR@10: 0.0517 (n=100,000)
reg= 0.05 -> Hit Rate@10: 0.1277 | MRR@10: 0.0519 (n=100,000)
reg=  0.1 -> Hit Rate@10: 0.1283 | MRR@10: 0.0520 (n=100,000)
reg=  0.2 -> Hit Rate@10: 0.1272 | MRR@10: 0.0519 (n=100,000)


regularization,hit_rate,mrr
f64,f64,f64
0.01,0.12544,0.051704
0.05,0.12765,0.051943
0.1,0.12834,0.052021
0.2,0.12721,0.051941


In [6]:
# Libreria implicit, Licencia MIT Copyright (c) 2016 Ben Frederickson
# https://github.com/benfred/implicit.git
from implicit.als import AlternatingLeastSquares
from threadpoolctl import threadpool_limits

# Limita BLAS a 1 hilo para evitar conflictos
threadpool_limits(1, "blas")


In [7]:
# Prueba de iteraciones
resultados_iter_finales = []
for iterations in [100, 150, 200]:
    m = AlternatingLeastSquares(factors=24, regularization=0.1, iterations=iterations, num_threads=0, random_state=42)
    m.fit(train_matrix, show_progress=True)
    hr, mrr, _ = evaluate(m, train_matrix, test_df, k=10)
    resultados_iter_finales.append({"iterations": iterations, "hit_rate": hr, "mrr": mrr})
    print(f"iterations={iterations:>4} -> Hit Rate@10: {hr:.4f} | MRR@10: {mrr:.4f}")

pl.DataFrame(resultados_iter_finales)

  0%|          | 0/100 [00:00<?, ?it/s]

iterations= 100 -> Hit Rate@10: 0.1440 | MRR@10: 0.0592


  0%|          | 0/150 [00:00<?, ?it/s]

iterations= 150 -> Hit Rate@10: 0.1434 | MRR@10: 0.0599


  0%|          | 0/200 [00:00<?, ?it/s]

iterations= 200 -> Hit Rate@10: 0.1442 | MRR@10: 0.0598


iterations,hit_rate,mrr
i64,f64,f64
100,0.144,0.059215
150,0.1434,0.059857
200,0.14415,0.059794


## 8. Modelo Final (ALS - Implicit)
Se procede a hacer el entrenamiento completo, con los siguientes hiperparámetros:
- Alpha: 32
- Factores: 24
- Regularización: 0.1
- Iteraciones: 100

In [7]:
# Libreria implicit, Licencia MIT Copyright (c) 2016 Ben Frederickson
# https://github.com/benfred/implicit.git
from implicit.als import AlternatingLeastSquares
from threadpoolctl import threadpool_limits

# Limita BLAS a 1 hilo para evitar conflictos
threadpool_limits(1, "blas")

# Modelo Final
BEST_ALPHA = 32
BEST_FACTORS = 24
BEST_REG = 0.1
ITERATIONS = 100

t0 = time.time()
train_matrix = build_csr(train_df, n_users, n_items, alpha=BEST_ALPHA)

model_final = AlternatingLeastSquares(
    factors=BEST_FACTORS,
    regularization=BEST_REG,
    iterations=ITERATIONS,
    num_threads=0,
    random_state=42,
    calculate_training_loss=True,
)
model_final.fit(train_matrix, show_progress=True)
print(f"\nEntrenamiento: {(time.time()-t0)/60:.1f} min")


  0%|          | 0/100 [00:00<?, ?it/s]


Entrenamiento: 50.6 min


In [11]:
hr_final, mrr_final, n_eval = evaluate(model_final, train_matrix, test_df, k=10, sample_users=100000)
print()
print(f"Modelo ALS Final    — Hit Rate@10: {hr_final:.4f} | MRR@10: {mrr_final:.4f} (n={n_eval:,})")
print(f"Popularidad         — Hit Rate@10: 0.1190 | MRR@10: 0.0563")


Modelo ALS Final    — Hit Rate@10: 0.1413 | MRR@10: 0.0576 (n=100,000)
Popularidad         — Hit Rate@10: 0.1190 | MRR@10: 0.0563


**Modelo ALS Final:**<br>
Evaluando Hit Rate, el 14.13% supera exitosamente el 11.90% de un Top 10 simple, y casi 4 puntos porcentuales mas que el Modelo 0<br>
Similar al calculo realizado antes, si comparamos con recomendaciones el azar, el modelo afinado pasa a ser:<br>
> 0.1413 / 0.0001276 ≈ 1,107 veces mejor que adivinar al azar.

Asimismo, el modelo final supera el simple Top 10 fijo de juegos, y en su lugar presenta recomendaciones personalizadas y, lo mas importante, no descarta el 99.999% de los juegos restantes...

Un detalle importante sobre el MRR, normalmente el calculo es sobre los aciertos, que tan arriba del top 10 se posicionaron.<br>
En este caso el calculo de MRR es global, es decir: se promedia sobre los N usuarios evaluados en total (no solo sobre los que tuvieron acierto) — los que no aciertan dentro del top-K aportan un 0 al promedio, no se excluyen del denominador.<br>

Si quisieramos obtener el MRR condicional (solo de los aciertos) la formula seria el MRR Global / Hit Rate:
> 0.0475 / 0.1060 → 0.4481

El inverso del MRR Condicional (1/0.4481) se obtiene 2.23 que representa la posicion promedio dentro del top 10 de juegos recomendados, cuando hay acierto.<br>
Esta metrica es bastante buena porque significa que cuando acertamos, la recomendacion esta entre los primeros lugares, y no al final de la lista.

In [15]:
# Guardar el modelo
MODEL_DIR = Path(r"models/Tunned")
MODEL_DIR.mkdir(exist_ok=True)

model_final.save(str(MODEL_DIR / "als_model.npz"))
print("Modelo guardado en:", MODEL_DIR / "als_model.npz")

# Para cargarlo después:
# from implicit.cpu.als import AlternatingLeastSquares

# model = AlternatingLeastSquares.load(str(MODEL_DIR / "als_model.npz"))

Modelo guardado en: models/Tunned/als_model.npz


Recommendaciones

In [16]:
item_lookup = dict(zip(item_map["item_idx"].to_list(), item_map["item_id"].to_list()))
user_lookup = dict(zip(user_map["user_idx"].to_list(), user_map["user_id"].to_list()))

sample_user_idx = int(train_df["user_idx"][0])
sample_user_id = user_lookup[sample_user_idx]

ids, scores = model_final.recommend(sample_user_idx, train_matrix[sample_user_idx], N=10)

print(f"Recomendaciones para usuario {sample_user_id} (idx={sample_user_idx}):\n")
for item_idx, score in zip(ids, scores):
    print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}")

Recomendaciones para usuario 76561197995052837 (idx=694711):

  app_id=1091500      score=1.1136
  app_id=292030       score=1.1134
  app_id=1245620      score=0.9823
  app_id=275850       score=0.9291
  app_id=1086940      score=0.8892
  app_id=306130       score=0.8311
  app_id=489830       score=0.8273
  app_id=1174180      score=0.8137
  app_id=440900       score=0.7611
  app_id=238960       score=0.7561


In [19]:
item_lookup = dict(zip(item_map["item_idx"].to_list(), item_map["item_id"].to_list()))
user_lookup = dict(zip(user_map["user_idx"].to_list(), user_map["user_id"].to_list()))
user_idx_lookup = {v: k for k, v in user_lookup.items()}
item_idx_lookup = {v: k for k, v in item_lookup.items()}

TARGET_USER_ID = 76561198324251650

if TARGET_USER_ID not in user_idx_lookup:
    print(f"El usuario {TARGET_USER_ID} no está en el subset de juegos (filtrado por DLC/herramientas")
    print("   o por tener menos de MIN_REVIEWS_POR_USUARIO_JUEGOS reviews de juegos).")
else:
    sample_user_idx = user_idx_lookup[TARGET_USER_ID]
    ids, scores = model_final.recommend(sample_user_idx, train_matrix[sample_user_idx], N=10)
    print(f"Recomendaciones para usuario {TARGET_USER_ID}:\n")
    for item_idx, score in zip(ids, scores):
        print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}")

Recomendaciones para usuario 76561198324251650:

  app_id=601150       score=0.3710
  app_id=203160       score=0.3385
  app_id=698780       score=0.3350
  app_id=107410       score=0.3293
  app_id=678960       score=0.3169
  app_id=814380       score=0.3049
  app_id=391220       score=0.2978
  app_id=637650       score=0.2919
  app_id=335300       score=0.2776
  app_id=1113000      score=0.2751


In [20]:
# Juegos similares a uno dado (útil para "usuarios que jugaron esto también jugaron...")
sample_item_idx = int(train_df["item_idx"][0])
sample_item_id = item_lookup[sample_item_idx]

similar_ids, similar_scores = model_final.similar_items(sample_item_idx, N=10)

print(f"Juegos similares a app_id={sample_item_id} (idx={sample_item_idx}):\n")
for item_idx, score in zip(similar_ids, similar_scores):
    print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}")

Juegos similares a app_id=435150 (idx=12614):

  app_id=435150       score=1.0000
  app_id=373420       score=0.7687
  app_id=455720       score=0.7445
  app_id=1086940      score=0.7239
  app_id=1705350      score=0.7047
  app_id=1184370      score=0.6913
  app_id=1565922      score=0.6810
  app_id=640820       score=0.6805
  app_id=560130       score=0.6687
  app_id=693820       score=0.6614


In [22]:
TARGET_APP_ID = 489830

if TARGET_APP_ID not in item_idx_lookup:
    print(f"El app_id={TARGET_APP_ID} no está en el subset de juegos.")
else:
    sample_item_idx = item_idx_lookup[TARGET_APP_ID]
    similar_ids, similar_scores = model_final.similar_items(sample_item_idx, N=10)
    print(f"Juegos similares a app_id={TARGET_APP_ID}:\n")
    for item_idx, score in zip(similar_ids, similar_scores):
        print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}")

Juegos similares a app_id=489830:

  app_id=489830       score=1.0000
  app_id=377160       score=0.8612
  app_id=1774380      score=0.8520
  app_id=22380        score=0.7919
  app_id=72850        score=0.7698
  app_id=900883       score=0.7645
  app_id=22330        score=0.7641
  app_id=2560000      score=0.7537
  app_id=1606670      score=0.7471
  app_id=913440       score=0.7294


In [5]:
# Libreria implicit, Licencia MIT Copyright (c) 2016 Ben Frederickson
# https://github.com/benfred/implicit.git
from implicit.cpu.als import AlternatingLeastSquares
from threadpoolctl import threadpool_limits

# Limita BLAS a 1 hilo para evitar conflictos
threadpool_limits(1, "blas")

MODEL_DIR = Path(r"models/Tunned")
model_final = AlternatingLeastSquares.load(str(MODEL_DIR / "als_model.npz"))

In [6]:
GAMES_PATH = Path(r"games/games.parquet")
GAMES_APPID_COL = "AppID"

games_lf = pl.scan_parquet(GAMES_PATH)
games_schema = games_lf.collect_schema()
assert GAMES_APPID_COL in games_schema, f"No encuentro '{GAMES_APPID_COL}'. Columnas: {list(games_schema.keys())}"

# AppIds únicos de games.parquet, casteados al mismo dtype que item_map["item_id"] -- evita que un
# mismatch de tipo (ej. Int64 vs Int32) genere un join silenciosamente vacío o casi vacío.
juegos_ids = (
    games_lf.select(pl.col(GAMES_APPID_COL).cast(item_map["item_id"].dtype).alias("item_id"))
      .unique()
      .collect()
)

# Semi-join: NO reindexa nada, solo filtra el item_map EXISTENTE del notebook 02 a los item_idx
# que también son juegos según games.parquet. Los índices que quedan son los mismos que ya usa
# `train_matrix` -- no hace falta traducir nada.
item_map_solo_juegos = item_map.join(juegos_ids, on="item_id", how="semi")
juegos_idx_en_modelo_original = item_map_solo_juegos["item_idx"].to_numpy()

print(f"Items totales en el catálogo (notebook 02): {item_map.height:,}")
print(f"Items que matchean con games.parquet:        {item_map_solo_juegos.height:,}")
print(f"Descartados (DLC/herramientas/no encontrados): {item_map.height - item_map_solo_juegos.height:,}")

Items totales en el catálogo (notebook 02): 78,376
Items que matchean con games.parquet:        58,638
Descartados (DLC/herramientas/no encontrados): 19,738


In [7]:
def evaluate_restricted(model, train_matrix, test_df, candidate_items, k=10, sample_users=20000, seed=42):
    test_pos = test_df.filter(pl.col("recommended") == 1)
    users_with_test = test_pos["user_idx"].unique().to_numpy()

    rng = np.random.default_rng(seed)
    if len(users_with_test) > sample_users:
        users_with_test = rng.choice(users_with_test, size=sample_users, replace=False)

    test_lookup = dict(zip(test_pos["user_idx"].to_list(), test_pos["item_idx"].to_list()))

    ids, _ = model_final.recommend(
        users_with_test,
        train_matrix[users_with_test],
        N=k,
        filter_already_liked_items=True,
        items=candidate_items,   # <- acá está la magia: solo rankea entre estos items
    )

    hits, rr = 0, []
    for row, uidx in enumerate(users_with_test):
        true_item = test_lookup[uidx]
        recs = ids[row]
        if true_item in recs:
            hits += 1
            rank = int(np.where(recs == true_item)[0][0]) + 1
            rr.append(1.0 / rank)
        else:
            rr.append(0.0)
    return hits / len(users_with_test), float(np.mean(rr)), len(users_with_test)


In [8]:
hr_filtrado, mrr_filtrado, n_eval = evaluate_restricted(
    model_final, train_matrix, test_df, juegos_idx_en_modelo_original, k=10
)
print(f"Modelo completo, candidatos=solo juegos -> Hit Rate@10: {hr_filtrado:.4f} | MRR@10: {mrr_filtrado:.4f}")

Modelo completo, candidatos=solo juegos -> Hit Rate@10: 0.1418 | MRR@10: 0.0589


### Recomendaciones/Similitudes (Filtrando solo juegos y un minimo de reviews)

In [13]:
MIN_REVIEWS_PARA_SIMILITUD = 100

n_reviews_por_item = (
    pl.concat([train_df.select(["item_idx"]), test_df.select(["item_idx"])])
      .group_by("item_idx").agg(pl.len().alias("n_reviews"))
)
n_reviews_lookup = dict(zip(n_reviews_por_item["item_idx"].to_list(), n_reviews_por_item["n_reviews"].to_list()))

item_lookup = dict(zip(item_map["item_idx"].to_list(), item_map["item_id"].to_list()))
user_lookup = dict(zip(user_map["user_idx"].to_list(), user_map["user_id"].to_list()))
user_idx_lookup = {v: k for k, v in user_lookup.items()}
item_idx_lookup = {v: k for k, v in item_lookup.items()}


In [15]:
TARGET_APP_ID = 489830

if TARGET_APP_ID not in item_idx_lookup:
    print(f"El app_id={TARGET_APP_ID} no está en el subset de juegos.")
else:
    sample_item_idx = item_idx_lookup[TARGET_APP_ID]
    similar_ids, similar_scores = model_final.similar_items(sample_item_idx, N=30)

    print(f"Juegos similares (solo juegos) a app_id={TARGET_APP_ID}:\n")
    mostrados = 0
    for item_idx, score in zip(similar_ids, similar_scores):
        n_rev = n_reviews_lookup.get(int(item_idx), 0)
        if n_rev < MIN_REVIEWS_PARA_SIMILITUD:
            continue
        print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}  (n_reviews={n_rev:,})")
        mostrados += 1
        if mostrados == 10:
            break

Juegos similares (solo juegos) a app_id=489830:

  app_id=489830       score=1.0000  (n_reviews=231,652)
  app_id=377160       score=0.8612  (n_reviews=284,791)
  app_id=22380        score=0.7919  (n_reviews=165,234)
  app_id=72850        score=0.7698  (n_reviews=261,706)
  app_id=900883       score=0.7645  (n_reviews=42,656)
  app_id=22330        score=0.7641  (n_reviews=42,656)
  app_id=306130       score=0.7080  (n_reviews=114,371)
  app_id=22370        score=0.7007  (n_reviews=37,548)
  app_id=1151340      score=0.6926  (n_reviews=81,491)
  app_id=22320        score=0.6491  (n_reviews=20,414)


#### Generar recomendaciones usando embeddings directamente.
En ALS, cada juego tiene un vector (item_factors) en el mismo espacio donde también viven los usuarios — así que es posible armar un "usuario sintético" promediando los vectores de los juegos de la lista, y rankear todo el catálogo contra ese perfil

In [16]:
TARGET_APP_IDS = [489830, 377160, 678960, 1091500, 582010, 524220, 883710]  # lista de juegos en lugar de uno solo

# Filtramos los que sí existen en el catálogo, avisando de los que no
item_idx_validos = []
for app_id in TARGET_APP_IDS:
    if app_id not in item_idx_lookup:
        print(f"⚠️  El app_id={app_id} no está en el subset de juegos, se ignora.")
    else:
        item_idx_validos.append(item_idx_lookup[app_id])

if not item_idx_validos:
    print("Ningún app_id de la lista está en el catálogo.")
else:
    # "Usuario sintético": el promedio de los embeddings de los juegos dados.
    # Es el mismo espacio vectorial que ocupan los usuarios reales, así que es un perfil
    # de preferencia coherente con lo que el modelo ya sabe rankear.
    pseudo_user_vector = model_final.item_factors[item_idx_validos].mean(axis=0)

    # Score de cada item del catálogo contra ese perfil sintético
    scores_todos = model_final.item_factors @ pseudo_user_vector

    # Orden descendente, pidiendo de más para poder filtrar después
    candidatos_idx = np.argsort(-scores_todos)[:200]

    print(f"Recomendaciones basadas en {len(item_idx_validos)} juego(s) de referencia:\n")
    mostrados = 0
    for item_idx in candidatos_idx:
        item_idx = int(item_idx)
        if item_idx in item_idx_validos:
            continue  # no recomendar de vuelta algo que ya está en la lista de entrada
        n_rev = n_reviews_lookup.get(item_idx, 0)
        if n_rev < MIN_REVIEWS_PARA_SIMILITUD:
            continue
        print(f"  app_id={item_lookup[item_idx]:<12} score={scores_todos[item_idx]:.4f}  (n_reviews={n_rev:,})")
        mostrados += 1
        if mostrados == 10:
            break

Recomendaciones basadas en 7 juego(s) de referencia:

  app_id=292030       score=0.3723  (n_reviews=609,560)
  app_id=698780       score=0.2737  (n_reviews=165,462)
  app_id=275850       score=0.2616  (n_reviews=243,156)
  app_id=72850        score=0.2558  (n_reviews=261,706)
  app_id=431960       score=0.2437  (n_reviews=556,090)
  app_id=271590       score=0.2352  (n_reviews=1,188,125)
  app_id=22380        score=0.2301  (n_reviews=165,234)
  app_id=1174180      score=0.2273  (n_reviews=380,493)
  app_id=306130       score=0.2180  (n_reviews=114,371)
  app_id=1151340      score=0.2166  (n_reviews=81,491)


### Modelo candidato a producción

Modelo final, con los mismos hiperparámetros, pero re-entrenado con todos los datos disponibles.

In [6]:
# Entrenamiento final con el 100% de los datos (train + test unidos), para el modelo de producción.
# Usa los hiperparámetros ya decididos para el Modelo 1 (alpha=32, factors=24, regularization=0.1, iterations=100).

# Libreria implicit, Licencia MIT Copyright (c) 2016 Ben Frederickson
# https://github.com/benfred/implicit.git
from implicit.als import AlternatingLeastSquares
from threadpoolctl import threadpool_limits

# Limita BLAS a 1 hilo para evitar conflictos
threadpool_limits(1, "blas")

df_completo = pl.concat([train_df, test_df])
print(f"Filas para entrenamiento final: {df_completo.height:,} (antes: train={train_df.height:,} + test={test_df.height:,})")

train_matrix_final = build_csr(df_completo, n_users, n_items, alpha=32.0)

model_produccion = AlternatingLeastSquares(
    factors=24,
    regularization=0.1,
    iterations=100,
    num_threads=0,
    random_state=42,
)

t0 = time.time()
model_produccion.fit(train_matrix_final, show_progress=True)
print(f"\nEntrenamiento completo en {(time.time()-t0)/60:.1f} min")


Filas para entrenamiento final: 92,958,418 (antes: train=75,909,345 + test=17,049,073)


  0%|          | 0/100 [00:00<?, ?it/s]


Entrenamiento completo en 50.1 min


In [7]:
# Guardar el modelo
MODEL_DIR = Path(r"models/Tunned")
MODEL_DIR.mkdir(exist_ok=True)

model_produccion.save(str(MODEL_DIR / "als_model_prod.npz"))
print("Modelo guardado en:", MODEL_DIR / "als_model_prod.npz")

Modelo guardado en: models/Tunned/als_model_prod.npz


#### Checks y valiadciones del modelo productivo

In [8]:
print(f"user_factors: {model_produccion.user_factors.shape} (esperado: ({n_users}, 24))")
print(f"item_factors: {model_produccion.item_factors.shape} (esperado: ({n_items}, 24))")

print(f"NaN en user_factors: {np.isnan(model_produccion.user_factors).sum()}")
print(f"NaN en item_factors: {np.isnan(model_produccion.item_factors).sum()}")
print(f"Inf en user_factors: {np.isinf(model_produccion.user_factors).sum()}")


user_factors: (17049073, 24) (esperado: (17049073, 24))
item_factors: (78376, 24) (esperado: (78376, 24))
NaN en user_factors: 0
NaN en item_factors: 0
Inf en user_factors: 0
